# 02 — Pré-processamento

Preparação das variáveis cadastrais e construção do alvo por cliente a partir do histórico de pagamentos.

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
RAW = Path("..") / "data" / "raw"
APPLICATION_FILE = RAW / "application_record.csv"
CREDIT_FILE = RAW / "credit_record.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "TARGET"

pd.set_option("display.max_columns", None)

In [23]:
app_df = pd.read_csv(APPLICATION_FILE)
credit_df = pd.read_csv(CREDIT_FILE)

app_renomear = {
    "CODE_GENDER": "GENERO",
    "FLAG_OWN_CAR": "POSSUI_CARRO",
    "FLAG_OWN_REALTY": "POSSUI_IMOVEL",
    "CNT_CHILDREN": "NUM_FILHOS",
    "AMT_INCOME_TOTAL": "RENDA_ANUAL",
    "NAME_INCOME_TYPE": "TIPO_RENDA",
    "NAME_EDUCATION_TYPE": "ESCOLARIDADE",
    "NAME_FAMILY_STATUS": "ESTADO_CIVIL",
    "NAME_HOUSING_TYPE": "TIPO_MORADIA",
    "DAYS_BIRTH": "IDADE_DIAS",
    "DAYS_EMPLOYED": "TEMPO_TRABALHO_DIAS",
    "FLAG_MOBIL": "POSSUI_CELULAR",
    "FLAG_WORK_PHONE": "TELEFONE_TRABALHO",
    "FLAG_PHONE": "TELEFONE",
    "FLAG_EMAIL": "POSSUI_EMAIL",
    "OCCUPATION_TYPE": "OCUPACAO",
    "CNT_FAM_MEMBERS": "TAMANHO_FAMILIA",
}
app_df = app_df.rename(columns=app_renomear)
credit_df = credit_df.rename(columns={"MONTHS_BALANCE": "MESES_HISTORICO"})

print(f"Aplicações: {app_df.shape}")
print(f"Histórico de crédito: {credit_df.shape}")
print("Colunas cadastrais:", app_df.columns.tolist())
print("Colunas de histórico:", credit_df.columns.tolist())

Aplicações: (438557, 18)
Histórico de crédito: (1048575, 3)
Colunas cadastrais: ['ID', 'GENERO', 'POSSUI_CARRO', 'POSSUI_IMOVEL', 'NUM_FILHOS', 'RENDA_ANUAL', 'TIPO_RENDA', 'ESCOLARIDADE', 'ESTADO_CIVIL', 'TIPO_MORADIA', 'IDADE_DIAS', 'TEMPO_TRABALHO_DIAS', 'POSSUI_CELULAR', 'TELEFONE_TRABALHO', 'TELEFONE', 'POSSUI_EMAIL', 'OCUPACAO', 'TAMANHO_FAMILIA']
Colunas de histórico: ['ID', 'MESES_HISTORICO', 'STATUS']


## 1. Conferindo dados faltantes e repetidos

A base de histórico tem várias linhas para cada cliente porque registra vários meses. Já um ID repetido na base cadastral pode indicar dois cadastros para a mesma pessoa; como não sei qual usar, vou retirar esses IDs repetidos.

In [24]:
nulos = app_df.isna().sum()
display(pd.DataFrame({"valores_ausentes": nulos, "percentual": (nulos / len(app_df) * 100).round(2)}).query("valores_ausentes > 0").sort_values("valores_ausentes", ascending=False))
print("IDs de cadastro repetidos:", app_df["ID"].duplicated().sum())
print("Ocupações não informadas:", app_df["OCUPACAO"].isna().sum())

,valores_ausentes,percentual
OCUPACAO,134203,30.6


IDs de cadastro repetidos: 47
Ocupações não informadas: 134203


**Decisão:** retiro os cadastros com `ID` repetido para não escolher um deles sem saber qual está correto. Ocupações vazias serão agrupadas em `Outros`. O tempo de trabalho vazio será preenchido durante o treino do modelo, sem consultar os dados de teste.

## 2. Como defini as classes do alvo

Marquei como `TARGET = 1` o cliente que teve atraso de 60 dias ou mais em algum mês (status 2, 3, 4 ou 5). `TARGET = 0` significa que não apareceu atraso de 60 dias ou mais no histórico observado. Essa classe ainda pode incluir clientes que tiveram atrasos menores; por isso, não quer dizer que nunca atrasaram.

In [25]:
credit_df["MAU_PAGADOR"] = credit_df["STATUS"].astype(str).isin(["2", "3", "4", "5"])
target_df = credit_df.groupby("ID", as_index=False)["MAU_PAGADOR"].max()
target_df["TARGET"] = target_df.pop("MAU_PAGADOR").astype(int)

app_df = app_df.loc[~app_df["ID"].duplicated(keep=False)].copy()
df = app_df.merge(target_df, on="ID", how="inner")
print("Distribuição do alvo:")
display(df["TARGET"].value_counts().rename_axis("TARGET").to_frame("clientes"))
print("Base após o merge:", df.shape)

Distribuição do alvo:


,clientes
TARGET,
0,35841
1,616


Base após o merge: (36457, 19)


## 3. Padronização para os modelos

Nesta etapa ainda não ajusto a escala estatística das variáveis. No notebook 03, o `StandardScaler` faz isso dentro do pipeline, depois da divisão entre treino e teste. A conversão de idade e tempo de trabalho de dias para anos acontece na seção seguinte deste notebook.

## 4. Ajustes nas variáveis

Converto idade e tempo de trabalho, que vieram em dias, para anos e deixo a unidade explícita nos nomes: `IDADE_ANOS` e `TEMPO_TRABALHO_ANOS`. O código especial `365243` no tempo de trabalho vira valor ausente; os outros valores válidos, inclusive de pensionistas, são mantidos. Ocupações vazias e pouco comuns são agrupadas em `Outros`. Os valores de filhos e tamanho da família são mantidos sem cortes. Retiro o indicador de celular porque é igual para todos os clientes.

In [26]:
df["OCUPACAO"] = df["OCUPACAO"].fillna("Outros")
df["IDADE_ANOS"] = df.pop("IDADE_DIAS").abs() / 365.25
df["TEMPO_TRABALHO_ANOS"] = (
    df.pop("TEMPO_TRABALHO_DIAS")
    .replace(365243, np.nan)
    .abs()
    / 365.25
)

frequencia_ocupacao = df["OCUPACAO"].value_counts(normalize=True)
ocupacoes_principais = frequencia_ocupacao[frequencia_ocupacao >= 0.03].index
df["OCUPACAO"] = df["OCUPACAO"].where(
    df["OCUPACAO"].isin(ocupacoes_principais), "Outros"
)
if df["POSSUI_CELULAR"].nunique(dropna=False) == 1:
    df = df.drop(columns="POSSUI_CELULAR")

display(df.head())
print("Nulos restantes:", int(df.isna().sum().sum()))
print("Idade em anos (resumo):")
display(df["IDADE_ANOS"].describe())
print("Tempo de trabalho em anos (resumo):")
display(df["TEMPO_TRABALHO_ANOS"].describe())

,ID,GENERO,POSSUI_CARRO,POSSUI_IMOVEL,NUM_FILHOS,RENDA_ANUAL,TIPO_RENDA,ESCOLARIDADE,ESTADO_CIVIL,TIPO_MORADIA,TELEFONE_TRABALHO,TELEFONE,POSSUI_EMAIL,OCUPACAO,TAMANHO_FAMILIA,TARGET,IDADE_ANOS,TEMPO_TRABALHO_ANOS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Outros,2.0,0,32.867899,12.435318
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Outros,2.0,0,32.867899,12.435318
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Outros,2.0,0,58.792608,3.104723
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,0,52.320329,8.353183
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,0,52.320329,8.353183


Nulos restantes: 6135
Idade em anos (resumo):


count    36457.000000
mean        43.737641
std         11.500479
min         20.503765
25%         34.119097
50%         42.609172
75%         53.218344
max         68.862423
Name: IDADE_ANOS, dtype: float64

Tempo de trabalho em anos (resumo):


count    30322.000000
mean         7.242994
std          6.454230
min          0.046543
25%          2.677618
50%          5.453799
75%          9.596167
max         43.019849
Name: TEMPO_TRABALHO_ANOS, dtype: float64

## 5. Salvando a base preparada

Salvo a base com nomes de colunas em português. Idade e tempo de trabalho já estão em anos e têm a unidade no nome da coluna. No notebook 03, as categorias são transformadas em números e os valores ausentes são preenchidos dentro do treino de cada modelo.

In [27]:
PROCESSED.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED / "dataset_tratado.csv"
df.to_csv(output_path, index=False)
print(f"Dataset salvo em {output_path} ({df.shape[0]} linhas, {df.shape[1]} colunas).")

Dataset salvo em ..\data\processed\dataset_tratado.csv (36457 linhas, 18 colunas).
